# 02 - Cleaning the production data

In the previous notebook I only looked at the data. Now I clean and
prepare the four production tables (plan, production, machine
stoppages, and material consumption), and calculate OEE (the plant's
efficiency).

I do everything in code instead of fixing it by hand in a spreadsheet,
for one simple reason: **it's repeatable**. If the data changes
tomorrow, I just run the notebook again and it's done -- no need to
repeat the manual work.

The functions I use here all live in `etl_lib.py` (in the same folder
as this notebook), so I'm not copying the same code across several
notebooks.


In [1]:
import re
import pandas as pd
import numpy as np
import etl_lib as etl

RAW = "../../datasets/raw"
DIM = "../../datasets/dim"
PROCESSED = "../../datasets/processed"


## Step 1: load the raw data

In [2]:
plan = pd.read_csv(f"{RAW}/fact_production_plan_raw.csv", encoding="utf-8-sig")
production = pd.read_csv(f"{RAW}/fact_production_raw.csv", encoding="utf-8-sig")
downtime = pd.read_csv(f"{RAW}/fact_downtime_raw.csv", encoding="utf-8-sig")
material_consumption = pd.read_csv(f"{RAW}/fact_material_consumption_raw.csv", encoding="utf-8-sig")
machine_capacity = pd.read_csv(f"{DIM}/dim_machine_setup.csv", encoding="utf-8-sig")

print("plan:", plan.shape)
print("production:", production.shape)
print("downtime:", downtime.shape)
print("material_consumption:", material_consumption.shape)


plan: (9128, 10)
production: (9137, 13)
downtime: (21874, 10)
material_consumption: (23504, 10)


## Step 2: generic cleaning

On all four tables I apply the same cleaning sequence:

1. `clean_disguised_blanks` -- turns `-`, `/`, a blank space, etc. into
   a real empty value (`NaN`).
2. `strip_extra_spaces` -- removes extra spaces at the start/end of text.
3. `standardize_categories` -- makes `INJECTION MOLDING` / `injection
   molding` all the same (`Injection Molding`).
4. `fix_negative_quantities` -- fixes negative numbers that shouldn't
   exist (you can't reject -12 pieces).
5. `drop_duplicate_rows` -- removes fully repeated rows.

I keep this inside a `clean_table` function so I don't repeat these 5
lines four times.


In [3]:
def clean_table(df, category_columns, quantity_columns, table_name):
    df = etl.clean_disguised_blanks(df)
    df = etl.strip_extra_spaces(df)
    df = etl.standardize_categories(df, category_columns)
    df = etl.fix_negative_quantities(df, quantity_columns)

    # RecordSeq is just a row number from the source system -- if I
    # included it in the duplicate check, two rows that are identical in
    # every way that matters would never be flagged as duplicates just
    # because they have a different RecordSeq
    columns_to_compare = [c for c in df.columns if c != "RecordSeq"]
    df, removed_count = etl.drop_duplicate_rows(df, columns=columns_to_compare)

    print(f"[{table_name}] removed {removed_count} duplicate rows -> {len(df):,} rows remain")
    return df

plan = clean_table(plan, ["Process"], [], "plan")
production = clean_table(production, ["Process"], ["RejectedQty"], "production")
downtime = clean_table(downtime, ["Process", "Shift"], [], "downtime")
material_consumption = clean_table(material_consumption, ["Process", "Shift"], ["EndWeightKg"], "material_consumption")


[plan] removed 18 duplicate rows -> 9,110 rows remain
[production] removed 27 duplicate rows -> 9,110 rows remain


[downtime] removed 65 duplicate rows -> 21,809 rows remain
[material_consumption] removed 70 duplicate rows -> 23,434 rows remain


## Step 3: fill in blank material lots

After the cleaning above, some `MaterialLot` values ended up blank.
This is almost always the operator forgetting to write it down -- not a
new lot starting there. So I fill it in with the last known lot for
that work order.


In [4]:
material_consumption["MaterialLot"] = etl.fill_blank_material_lot(material_consumption)

print("Blank values in MaterialLot after filling:", material_consumption["MaterialLot"].isna().sum())


Blank values in MaterialLot after filling: 0


## Step 4: calendar and shift columns

For all four tables, I add the ISO week, the ISO weekday, and the shift
number (computed from the start time, or read straight from the
`Shift` column when it already exists).


In [5]:
for table in (plan, production, downtime, material_consumption):
    calendar_columns = etl.add_calendar_columns(table, "Date")
    table["ISOWeek"] = calendar_columns["ISOWeek"]
    table["ISOWeekday"] = calendar_columns["ISOWeekday"]

SHIFT_MAP = {"Shift 1": 1, "Shift 2": 2, "Shift 3": 3}

plan["ShiftNumber"] = etl.compute_shift_number(plan["StartTime"])
production["ShiftNumber"] = etl.compute_shift_number(production["StartTime"])
downtime["ShiftNumber"] = downtime["Shift"].map(SHIFT_MAP)
material_consumption["ShiftNumber"] = material_consumption["Shift"].map(SHIFT_MAP)


## Step 5: the `LotId` traceability code

The `LotId` is 16 characters long and combines several pieces of
information: year, week, weekday, shift, process, machine, work order,
and a material-lot sequence number. This is the code that later lets
any quality inspection be traced back to the exact work order and raw
material lot that produced it.

First I compute the material lot sequence (the last 2 digits), then I
build the full `LotId` -- first for the material consumption table,
then for the production table (which uses the sequence from the first
material row of each order).


In [6]:
material_consumption["MaterialLotSeq"] = etl.compute_material_lot_sequence(material_consumption)

material_consumption["LotIdPrefix"] = etl.build_lotid_prefix(
    material_consumption["Date"], material_consumption["ShiftNumber"], material_consumption["Process"],
    material_consumption["MachineId"], material_consumption["WorkOrder"]
)
material_consumption["LotIdStart"] = material_consumption["LotIdPrefix"] + material_consumption["MaterialLotSeq"].astype(str).str.zfill(2)

material_consumption.head()


,RecordSeq,Date,Process,MachineId,Shift,WorkOrder,MaterialId,MaterialLot,StartWeightKg,EndWeightKg,ISOWeek,ISOWeekday,ShiftNumber,MaterialLotSeq,LotIdPrefix,LotIdStart
0,14577,2026-03-02,Hot Foil Stamping,HF-001,Shift 1,WO-1314,COR-001,COL-COR-001-107,59.11,34.24,10,1,1,1,26101150101314,2610115010131401
1,4737,2025-09-30,Injection Molding,IM-003,Shift 3,WO-3119,COR-007,COL-COR-007-605,390.89,223.88,40,2,3,2,25402320303119,2540232030311902
2,12421,2026-01-27,Screen Printing,SS-001,Shift 3,WO-9360,COR-003,COL-COR-003-2297,373.20,275.14,5,2,3,4,26052340109360,2605234010936004
3,7066,2025-11-05,Blow Molding,ISBM-003,Shift 1,WO-6239,COR-008,COL-COR-008-3402,12.73,10.47,45,3,1,2,25453110306239,2545311030623902
4,21517,2026-06-17,Blow Molding,ISBM-005,Shift 3,WO-7554,COR-005,COL-COR-005-1833,40.99,20.79,25,3,3,4,26253310507554,2625331050755404


In [7]:
production["LotIdPrefix"] = etl.build_lotid_prefix(
    production["Date"], production["ShiftNumber"], production["Process"],
    production["MachineId"], production["WorkOrder"]
)

# takes the sequence from the FIRST material record of each work order
first_sequence_per_order = (
    material_consumption.sort_values(["WorkOrder", "RecordSeq"])
    .groupby("WorkOrder")["MaterialLotSeq"]
    .first()
)
production["MaterialLotSeq"] = production["WorkOrder"].map(first_sequence_per_order).fillna(1).astype(int)
production["LotId"] = production["LotIdPrefix"] + production["MaterialLotSeq"].astype(str).str.zfill(2)

production[["WorkOrder", "LotId"]].head()


,WorkOrder,LotId
0,WO-1703,2549415020170301
1,WO-8533,2621221070853301
2,WO-6123,2533231030612301
3,WO-5594,2531331020559401
4,WO-2703,2549112020270301


## Step 6: exact start and end time for each order

The raw tables only store the time of day (`StartTime`), not the full
date and time. I need this to later figure out which order was running
when each machine stoppage happened.


In [8]:
production = production.merge(plan[["WorkOrder", "PlannedHours"]], on="WorkOrder", how="left")

production["_start_dt"] = pd.to_datetime(production["Date"]) + pd.to_timedelta(production["StartTime"].astype(str))
production["_end_dt"] = production["_start_dt"] + pd.to_timedelta(production["PlannedHours"].astype(float), unit="h")


## Step 7: maintenance info, and which order was running during each stoppage

First I compute the duration of each stoppage and whether it was a
genuine unplanned failure. Then, for each stoppage, I look up which
work order was running at that time, on the same machine.


In [9]:
downtime = etl.add_maintenance_info(downtime)
downtime["LeadTimeProdHours"] = downtime["DowntimeDurationMin"] / 60.0

downtime["MatchedWorkOrder"] = etl.find_work_order_for_stoppage(downtime, production)

match_rate = downtime["MatchedWorkOrder"].notna().mean()
print(f"Stoppages matched to a work order: {100*match_rate:.1f}%")
print("(the rest are stoppages logged in a gap between two orders, or right at the start/end of the time period)")


Stoppages matched to a work order: 84.8%
(the rest are stoppages logged in a gap between two orders, or right at the start/end of the time period)


## Step 8: rated capacity of each machine

I need to know how many pieces per hour each machine+mold combination
should produce, to later compute Performance (part of OEE). That
information lives in the `dim_machine_setup` table, except for screen
printing and hot foil stamping, which use fixed values.


In [10]:
def capacity_as_number(text):
    # the text comes in like "~1200 pieces/h" -- I just want the number (1200)
    digits_only = re.sub(r"[^\d]", "", str(text).split("/")[0])
    return int(digits_only)

capacity_by_machine = {}
for _, row in machine_capacity.iterrows():
    key = (row["MachineId"], row["MoldId"])
    capacity_by_machine[key] = capacity_as_number(row["RatedCapacityPerHour"])

# screen printing and hot foil stamping aren't in dim_machine_setup -- use
# the fixed values agreed on for those two processes
fixed_capacities = {"SS-001": 1025, "SS-002": 950, "HF-001": 700, "HF-002": 650}
for machine, capacity in fixed_capacities.items():
    tools_on_this_machine = production.loc[production["MachineId"] == machine, "ToolId"].unique()
    for tool in tools_on_this_machine:
        capacity_by_machine[(machine, tool)] = capacity


## Step 9: calculate OEE

Now the main calculation: Availability, Performance, Quality, and OEE
(which is the product of the three).


In [11]:
production["LeadTimeProdHours"] = etl.compute_production_time(production, duration_hours_column="PlannedHours")

# only use stoppages that were matched to a work order, and rename the
# column to match what the OEE function expects
downtime_with_order = downtime.dropna(subset=["MatchedWorkOrder"]).rename(columns={"MatchedWorkOrder": "WorkOrder"})

production_without_planned_hours = production.drop(columns=["PlannedHours"])
production = etl.compute_oee_components(production_without_planned_hours, plan, downtime_with_order, capacity_by_machine)

production[["WorkOrder", "Availability", "Performance", "Quality", "OEE"]].head()


,WorkOrder,Availability,Performance,Quality,OEE
7788,WO-1001,0.747726,0.920164,0.997834,0.686540
5100,WO-1002,1.000000,0.906660,0.997934,0.904787
4478,WO-1003,1.000000,0.900320,0.998127,0.898634
4567,WO-1004,1.000000,0.930660,0.998205,0.928989
5067,WO-1005,1.000000,0.925100,0.998520,0.923731


In [12]:
print("Plant-wide average:")
print("  Availability:", round(production["Availability"].mean(), 3))
print("  Performance: ", round(production["Performance"].mean(), 3))
print("  Quality:     ", round(production["Quality"].mean(), 3))
print("  OEE:         ", round(production["OEE"].mean(), 3))
print()
print("Average OEE by process:")
print(production.groupby("Process")["OEE"].mean().round(3))


Plant-wide average:
  Availability: 0.792
  Performance:  0.897
  Quality:      0.976
  OEE:          0.693

Average OEE by process:
Process
Blow Molding         0.667
Hot Foil Stamping    0.781
Injection Molding    0.671
Screen Printing      0.770
Name: OEE, dtype: float64


The plant's average OEE is around 69%. Availability is the weakest of
the three components, which points to unplanned downtime as the main
opportunity for improvement -- this shows up again throughout the
analysis notebooks.


## Step 10: drop helper columns and save

A few columns only existed to help with the calculations above
(`_start_dt`, `_end_dt`) and have no further use once `LotId` and OEE
are computed -- so I drop them before saving, to avoid carrying dead
weight into MySQL and Power BI later.


In [13]:
helper_columns = ["_start_dt", "_end_dt", "_next_start_dt", "LotIdPrefix"]
# LotIdPrefix is also just a helper column (it only exists to be glued
# to the sequence number into LotId / LotIdStart above) -- same
# reasoning as _start_dt/_end_dt, so it's dropped here too instead of
# leaking into the final CSVs and the database

plan.to_csv(f"{PROCESSED}/fact_production_plan_processed.csv", encoding="utf-8-sig", index=False)

final_production = production.drop(columns=[c for c in helper_columns if c in production.columns])
final_production.to_csv(f"{PROCESSED}/fact_production_processed.csv", encoding="utf-8-sig", index=False)

downtime.to_csv(f"{PROCESSED}/fact_downtime_processed.csv", encoding="utf-8-sig", index=False)

final_material_consumption = material_consumption.drop(
    columns=[c for c in helper_columns + ["RecordSeq"] if c in material_consumption.columns]
)
final_material_consumption.to_csv(f"{PROCESSED}/fact_material_consumption_processed.csv", encoding="utf-8-sig", index=False)

print("Files saved to", PROCESSED)


Files saved to ../../datasets/processed
